# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata  # metadata is an object
print(f"Dataset name: {meta.name}\n\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s and related columns.

Every entity (record set, field, or column) is referenced by its unique `@id`.

In [ ]:
# List record sets in the dataset with their @ids and field information
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
        for c in field.columns:
            print(f"        * Column: {c.name} (@id: {c.id})")
    print()

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrames for analysis.

Use the record set and field `@id`s from the overview above to ensure correct referencing.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for rec_id in record_set_ids:
    recs = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(recs)
    dataframes[rec_id] = df
    print(f"Loaded record set {rec_id} with {df.shape[0]} records and columns: {df.columns.tolist()}")

# Choose the first available record set for further analysis
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
# Show sample rows if available
if main_record_set_id is not None:
    print(f"\nFirst 5 rows of record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, or grouping by field keys.

> *All fields are referenced by their `@id`. Please check the overview for exact IDs and field types.*

In [ ]:
import numpy as np
# You must set these variables to match one of the numeric fields in your dataset.
# For example, using field @id of a numeric column such as 'coverage' or 'molecular_weight'
record_set_id = main_record_set_id  # Use the chosen main record set
df = dataframes[record_set_id] if record_set_id is not None else pd.DataFrame()

# --- Update the below if you know which field corresponds to a numeric column ---
# Example selection. Please CHANGE these IDs to match your dataset's actual field @ids.

# Let's try to automatically pick a numeric field (as an example: look for 'coverage', 'Abundance', 'MW', etc.)
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.int64, float, int]]
if not possible_numeric_fields and len(df.columns) > 0:
    # If all fields are strings, try casting one to numeric
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if df[c].dtype in [np.float64, np.int64]:
                possible_numeric_fields.append(c)
                break
        except Exception:
            continue
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]  # Use the first detected numeric column
else:
    numeric_field_id = None

if numeric_field_id:
    # Filter values above a threshold (example: values > 10)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to pick a group field (typically a categorical field)
    group_field = None
    for c in df.columns:
        if df[c].dtype == object and c != numeric_field_id:
            group_field = c
            break
    if group_field:
        # Note: mean only applies to numeric columns; this is exemplary
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA. Please inspect columns above and select a numeric field @id.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we demonstrate a histogram and a scatter plot (if there are multiple numeric fields).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Attempt to make a scatter plot if another numeric field is available
    more_numeric = [c for c in df.columns if c != numeric_field_id and pd.api.types.is_numeric_dtype(df[c])]
    if more_numeric:
        plt.figure(figsize=(6,6))
        plt.scatter(df[numeric_field_id], df[more_numeric[0]], alpha=0.5)
        plt.xlabel(numeric_field_id)
        plt.ylabel(more_numeric[0])
        plt.title(f"Scatter plot of {numeric_field_id} vs {more_numeric[0]}")
        plt.show()
else:
    print("No suitable numeric field for visualization. Please adjust your field selection.")

## 6. Conclusion
In this notebook, you loaded the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using `mlcroissant`, inspected its structure, and loaded actual data into Pandas DataFrames for exploration.

You performed simple exploratory analyses and visualizations, referencing all fields and record sets via their stable `@id`s as best practice. For rich, domain-specific insights, consult the manuscript or data dictionary and adjust field selections and filtering accordingly.